# S003 — Per-Ticker Drilldown

**Set `TICKER` in cell 2, then Run All.**

Covers: price + trade events · yield rank vs EPS rank · composite score · weight history · daily contribution · trade log · cumulative P&L attribution.

In [ ]:
# ════════════════════════════════════════════════
TICKER = "HSBA"   # change and Run All
# ════════════════════════════════════════════════
import sys, os, warnings, json
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

RUN_AT = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")

sys.path.insert(0, str(Path(r"c:\Personal\Business & Investments\Python codes")))
from signum import Chart
from signum.engine.dashboard import Dashboard
from signum.engine.statchart import StatChart

def _find_btest_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "AGENT_DSL_REFERENCE.md").exists(): return p
    return Path(r"c:\Personal\Business & Investments\Python codes\btest")

BTEST_ROOT  = _find_btest_root()
os.chdir(BTEST_ROOT)

SIGNAL_ROOT = Path("research/generated/Dividend Growth/signals/003_lse_div_eps")
OUTPUTS     = SIGNAL_ROOT / "outputs"
DATA_DIR    = SIGNAL_ROOT / "data"
SHARED_DATA = Path("research/generated/Dividend Growth/shared_data")

weights   = pd.read_parquet(OUTPUTS / "weights.parquet")
trades    = pd.read_parquet(OUTPUTS / "trades.parquet")
equity    = pd.read_parquet(OUTPUTS / "equity.parquet")
composite = pd.read_parquet(DATA_DIR / "composite.parquet")

for df_ in [equity, weights, composite]:
    if hasattr(df_.index, "tz") and df_.index.tz is not None:
        df_.index = df_.index.tz_localize(None)

eq_col = next((c for c in equity.columns if any(k in c.lower() for k in ("nav","portfolio","equity","total"))), equity.columns[0])
eq = equity[eq_col].dropna()

prices_long = pd.read_parquet(SHARED_DATA / "lse_prices.parquet")
t_price = prices_long[prices_long["ticker"]==TICKER].set_index("date")["close"].sort_index()
if hasattr(t_price.index, "tz") and t_price.index.tz is not None:
    t_price.index = t_price.index.tz_localize(None)

t_weight  = weights[TICKER].fillna(0) if TICKER in weights.columns else pd.Series(dtype=float)
t_weight.index = pd.to_datetime(t_weight.index).normalize()
t_trades  = trades[trades["instrument"]==TICKER].copy()
t_comp    = composite[TICKER].dropna() if TICKER in composite.columns else pd.Series(dtype=float)

print(f"Ticker: {TICKER} | Price rows: {len(t_price)} | Days held: {(t_weight>0.001).sum()} | Trades: {len(t_trades)}")


---
## 1 · Price History + Trade Events

In [ ]:
price_df = pd.DataFrame({"time": t_price.index, "value": t_price.values})

# Hold-period shading
hold_ser = t_weight.reindex(t_price.index, method="ffill").fillna(0)
hold_df  = pd.DataFrame({"time": hold_ser.index, "position": (hold_ser > 0.001).astype(int)})

chart = Chart(height=340, theme="dark", watermark=f"{TICKER} — Close Price (GBX)")
chart.line(price_df, name="Close", color="#90caf9")
if hold_df["position"].any():
    chart.shade(hold_df, position_col="position")

# Buy/sell markers
if len(t_trades):
    buy_times  = pd.to_datetime(t_trades.loc[t_trades["side"].str.upper()=="BUY",  "datetime"]).dt.normalize()
    sell_times = pd.to_datetime(t_trades.loc[t_trades["side"].str.upper()=="SELL", "datetime"]).dt.normalize()
    buy_prices  = t_price.reindex(buy_times,  method="nearest").values
    sell_prices = t_price.reindex(sell_times, method="nearest").values

    if len(buy_times):
        buys = pd.DataFrame({"time": buy_times.values,  "value": buy_prices})
        chart.marker(buys, shape="arrow_up",   color="#26a69a", text="Buy")
    if len(sell_times):
        sells = pd.DataFrame({"time": sell_times.values, "value": sell_prices})
        chart.marker(sells, shape="arrow_down", color="#ef5350", text="Sell")

chart


---
## 2 · Yield Rank vs EPS Growth Rank Over Time

In [ ]:
# Load sub-components if available
try:
    yield_rank = pd.read_parquet(DATA_DIR / "yield_rank.parquet")
    eps_rank   = pd.read_parquet(DATA_DIR / "eps_rank.parquet")
    for df_ in [yield_rank, eps_rank]:
        if hasattr(df_.index, "tz") and df_.index.tz is not None:
            df_.index = df_.index.tz_localize(None)
    t_yield = yield_rank[TICKER].dropna() if TICKER in yield_rank.columns else pd.Series(dtype=float)
    t_eps   = eps_rank[TICKER].dropna()   if TICKER in eps_rank.columns   else pd.Series(dtype=float)
    has_sub = len(t_yield) > 0 or len(t_eps) > 0
except FileNotFoundError:
    has_sub = False
    t_yield = pd.Series(dtype=float)
    t_eps   = pd.Series(dtype=float)

if has_sub:
    yr_df  = pd.DataFrame({"time": t_yield.index, "value": t_yield.values})
    eps_df = pd.DataFrame({"time": t_eps.index,   "value": t_eps.values})
    Dashboard(
        panes=[
            Chart(height=160).line(yr_df,  name="Yield rank (cross-sectional)",     color="#26a69a"),
            Chart(height=160).line(eps_df, name="EPS growth rank (cross-sectional)", color="#ffa726"),
        ],
        titles=[
            f"{TICKER} — Yield Rank (percentile within universe)",
            f"{TICKER} — EPS Growth Rank (percentile within universe)",
        ],
        theme="dark",
    )
else:
    # Fallback: try to split composite into 2 halves as proxy
    t_comp_df = pd.DataFrame({"time": t_comp.index, "value": t_comp.values})
    Chart(height=220, theme="dark", watermark=f"{TICKER} — Composite Score (sub-signals not separately stored)").line(
        t_comp_df, name="Composite", color="#26a69a"
    )


---
## 3 · Composite Score + Threshold

In [ ]:
comp_df = pd.DataFrame({"time": t_comp.index, "value": t_comp.values})
chart_comp = Chart(height=240, theme="dark", watermark=f"{TICKER} — Composite Score (z-score)")
chart_comp.line(comp_df, name="Composite score", color="#26a69a")
chart_comp


---
## 4 · Portfolio Weight History

In [ ]:
wt_df = pd.DataFrame({"time": t_weight.index, "value": t_weight.values * 100})
Chart(height=200, theme="dark", watermark=f"{TICKER} — Portfolio Weight (%)").area(
    wt_df, name="Weight %", color="#1976d2"
)


---
## 5 · Daily Return Contribution

In [ ]:
price_ret_tk = t_price.pct_change(fill_method=None).clip(-0.5, 0.5)
t_wt_aligned = t_weight.reindex(price_ret_tk.index, method="ffill").fillna(0)
daily_contrib = t_wt_aligned.shift(1).fillna(0) * price_ret_tk * 10000  # bps

cum_contrib = daily_contrib.cumsum()

dc_df  = pd.DataFrame({"time": daily_contrib.index, "value": daily_contrib.values})
cc_df  = pd.DataFrame({"time": cum_contrib.index,   "value": cum_contrib.values})

Dashboard(
    panes=[
        Chart(height=180).histogram(dc_df, name="Daily contribution (bps)", color="#26a69a"),
        Chart(height=160).area(cc_df, name="Cumulative contribution (bps)", color="#1976d2"),
    ],
    titles=[
        f"{TICKER} — Daily Return Contribution (bps)",
        f"{TICKER} — Cumulative Contribution  ·  Total: {cum_contrib.iloc[-1]:.0f} bps",
    ],
    theme="dark",
)


---
## 6 · Trade Log

In [ ]:
if len(t_trades):
    tlog = t_trades.copy()
    tlog["datetime"] = pd.to_datetime(tlog["datetime"]).dt.normalize()
    tlog = tlog.sort_values("datetime")
    display(tlog.style
        .format({c: "{:.4f}" for c in tlog.select_dtypes("float").columns})
        .set_caption(f"{TICKER} — All Trades ({len(tlog)} total)")
        .hide(axis="index"))
else:
    print(f"No trades found for {TICKER}")


---
## 7 · Cumulative P&L

In [ ]:
Dashboard(
    panes=[
        Chart(height=220).area(
            cc_df.rename(columns={"value":"value"}),
            name="Cumulative contribution (bps, attribution method)", color="#26a69a"),
    ],
    titles=[f"{TICKER} — Cumulative Return Attribution  ·  Final: {cum_contrib.dropna().iloc[-1] if len(cum_contrib.dropna()) else 0:.0f} bps"],
    theme="dark",
)
